# 📌 Topic 3: Building a Neural Network from Scratch in NumPy
> **Deep Learning Crash Course — Part 3**

In this notebook, we will build a 2-layer Neural Network **completely from scratch using pure NumPy** (no PyTorch, no TensorFlow).

We will write:
1. **Weight & Bias Initialization**.
2. **Forward Propagation** (Input $\rightarrow$ Hidden Layer $\rightarrow$ Output).
3. **Backpropagation** (Calculating gradients via Chain Rule).
4. **Gradient Descent Updates** & Decision Boundary Plotting.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

# Set random seed
np.random.seed(42)

# Generate Non-Linear Moons Dataset
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
y = y.reshape(-1, 1)

class NeuralNetworkFromScratch:
    def __init__(self, input_dim=2, hidden_dim=16, output_dim=1):
        # Weight Initialization (He Initialization)
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros((1, hidden_dim))
        
        self.W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(1.0 / hidden_dim)
        self.b2 = np.zeros((1, output_dim))
        
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    
    def forward(self, X):
        # Hidden Layer (ReLU Activation)
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = np.maximum(0, self.z1)
        
        # Output Layer (Sigmoid Activation)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self.sigmoid(self.z2)
        return self.a2
    
    def backward(self, X, y, lr=0.5):
        m = X.shape[0]
        
        # 1. Output Layer Gradient
        dz2 = self.a2 - y
        dW2 = np.dot(self.a1.T, dz2) / m
        db2 = np.sum(dz2, axis=0, keepdims=True) / m
        
        # 2. Hidden Layer Gradient (Chain Rule)
        da1 = np.dot(dz2, self.W2.T)
        dz1 = da1 * (self.z1 > 0) # ReLU Derivative
        dW1 = np.dot(X.T, dz1) / m
        db1 = np.sum(dz1, axis=0, keepdims=True) / m
        
        # 3. Update Weights & Biases
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        
    def compute_loss(self, y_pred, y_true):
        eps = 1e-15
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Train the Network
model = NeuralNetworkFromScratch(input_dim=2, hidden_dim=16, output_dim=1)
loss_history = []

for epoch in range(1000):
    preds = model.forward(X)
    loss = model.compute_loss(preds, y)
    loss_history.append(loss)
    model.backward(X, y, lr=0.5)

print(f"Final Loss after 1000 epochs: {loss_history[-1]:.4f}")

# Visualizing Decision Boundary
xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1.0, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
probs = model.forward(grid).reshape(xx.shape)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(loss_history, color="purple", lw=2)
plt.title("Training Loss Curve (NumPy NN)")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")

plt.subplot(1, 2, 2)
plt.contourf(xx, yy, probs, levels=20, cmap="Spectral", alpha=0.8)
plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="Spectral", edgecolors="k")
plt.title("Decision Boundary Learned by NumPy Neural Net")
plt.tight_layout()
plt.show()
